# WiFi CSI Signal Exploration Notebook

> **Important note:** This notebook uses synthetic CSI-like data only. It demonstrates a research-prototype workflow for WiFi CSI fall detection, clinical-safety-aware metrics, and synthetic adversarial robustness stress testing. The results should not be interpreted as real-world, clinical, hardware-validated, or medical-grade fall-detection performance.

## Notebook goals

This notebook demonstrates a simple end-to-end prototype pipeline:

1. Generate synthetic CSI-like signals
2. Visualize amplitude and phase behavior
3. Simulate normal activity and fall-like events
4. Preprocess signals
5. Extract simple features
6. Train a baseline classifier
7. Evaluate prototype results
8. Add clinical-safety-aware metrics
9. Run synthetic adversarial robustness stress tests
10. Save figures and summaries

This notebook supports the broader research direction of secure WiFi CSI sensing for healthcare-oriented monitoring.

## 1. Import libraries

The notebook uses common scientific Python libraries: NumPy, pandas, matplotlib, SciPy, and scikit-learn.

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from scipy.ndimage import uniform_filter1d
from scipy.fft import rfft, rfftfreq

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, classification_report

RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)

os.makedirs('../figures', exist_ok=True)
os.makedirs('../results', exist_ok=True)

print('Libraries imported successfully.')

## 2. Generate synthetic CSI-like data

In a real WiFi sensing system, Channel State Information describes how WiFi signals are affected by the surrounding environment. Human movement can change CSI amplitude and phase patterns.

Here, we generate simplified synthetic CSI-like signals. A normal activity sample has smoother variations, while a fall-like event includes a short sudden disturbance.

In [ ]:
def generate_normal_activity_csi(num_packets=300, num_subcarriers=30, noise_level=0.05):
    """Generate synthetic CSI-like amplitude and phase for normal activity."""
    t = np.linspace(0, 1, num_packets)
    amplitude = np.zeros((num_packets, num_subcarriers))
    phase = np.zeros((num_packets, num_subcarriers))

    for sc in range(num_subcarriers):
        base_amp = 1.0 + 0.1 * np.sin(2 * np.pi * (2 + sc * 0.02) * t)
        slow_motion = 0.05 * np.sin(2 * np.pi * (5 + sc * 0.01) * t)
        amplitude[:, sc] = base_amp + slow_motion + noise_level * np.random.randn(num_packets)

        base_phase = 0.2 * np.sin(2 * np.pi * (3 + sc * 0.02) * t)
        phase[:, sc] = base_phase + noise_level * np.random.randn(num_packets)

    return amplitude, phase


def generate_fall_event_csi(num_packets=300, num_subcarriers=30, noise_level=0.05):
    """Generate synthetic CSI-like amplitude and phase with a fall-like disturbance."""
    amplitude, phase = generate_normal_activity_csi(num_packets, num_subcarriers, noise_level)

    event_center = int(num_packets * 0.55)
    event_width = int(num_packets * 0.06)
    event_window = np.arange(num_packets)
    disturbance = np.exp(-0.5 * ((event_window - event_center) / event_width) ** 2)

    for sc in range(num_subcarriers):
        scale = 0.4 + 0.2 * np.sin(sc)
        amplitude[:, sc] += scale * disturbance
        phase[:, sc] += 0.5 * scale * disturbance * np.sin(2 * np.pi * 10 * event_window / num_packets)

    return amplitude, phase


normal_amp, normal_phase = generate_normal_activity_csi()
fall_amp, fall_phase = generate_fall_event_csi()

print('Normal amplitude shape:', normal_amp.shape)
print('Fall-like amplitude shape:', fall_amp.shape)

## 3. Visualize CSI amplitude

The plot below compares the average CSI amplitude across subcarriers for normal activity and a synthetic fall-like event.

In [ ]:
time_axis = np.arange(normal_amp.shape[0])

plt.figure(figsize=(10, 5))
plt.plot(time_axis, normal_amp.mean(axis=1), label='Normal activity')
plt.plot(time_axis, fall_amp.mean(axis=1), label='Fall-like event')
plt.xlabel('Packet index')
plt.ylabel('Mean CSI amplitude')
plt.title('Synthetic CSI Amplitude: Normal vs Fall-like Event')
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.savefig('../figures/synthetic_csi_amplitude.png', dpi=200)
plt.show()

## 4. Visualize CSI phase

The plot below compares the average CSI phase across subcarriers for normal activity and a synthetic fall-like event.

In [ ]:
plt.figure(figsize=(10, 5))
plt.plot(time_axis, normal_phase.mean(axis=1), label='Normal activity')
plt.plot(time_axis, fall_phase.mean(axis=1), label='Fall-like event')
plt.xlabel('Packet index')
plt.ylabel('Mean CSI phase')
plt.title('Synthetic CSI Phase: Normal vs Fall-like Event')
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.savefig('../figures/synthetic_csi_phase.png', dpi=200)
plt.show()

## 5. Compare one subcarrier

A single subcarrier view helps show how a sudden fall-like disturbance can appear as a localized change in the CSI time series.

In [ ]:
subcarrier_index = 10

plt.figure(figsize=(10, 5))
plt.plot(time_axis, normal_amp[:, subcarrier_index], label='Normal activity')
plt.plot(time_axis, fall_amp[:, subcarrier_index], label='Fall-like event')
plt.xlabel('Packet index')
plt.ylabel('CSI amplitude')
plt.title(f'Example Subcarrier {subcarrier_index}: Normal vs Fall-like Event')
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.savefig('../figures/fall_vs_nonfall_example.png', dpi=200)
plt.show()

## 6. Preprocess synthetic CSI signals

This simple preprocessing step removes the mean, normalizes the signal, and applies a moving-average smoothing filter.

In [ ]:
def remove_mean(signal):
    """Remove the mean from each subcarrier."""
    return signal - np.mean(signal, axis=0, keepdims=True)


def normalize_csi(signal):
    """Normalize each subcarrier to zero mean and unit standard deviation."""
    mean = np.mean(signal, axis=0, keepdims=True)
    std = np.std(signal, axis=0, keepdims=True) + 1e-8
    return (signal - mean) / std


def smooth_signal(signal, window_size=5):
    """Apply moving-average smoothing along the packet/time axis."""
    return uniform_filter1d(signal, size=window_size, axis=0)


normal_amp_processed = smooth_signal(normalize_csi(remove_mean(normal_amp)))
fall_amp_processed = smooth_signal(normalize_csi(remove_mean(fall_amp)))

print('Preprocessing complete.')

## 7. Feature extraction

The next step extracts simple time-domain and frequency-domain features from each synthetic CSI sample.

In [ ]:
def extract_features(amplitude, sampling_rate=100):
    """Extract simple features from a CSI amplitude matrix."""
    mean_over_subcarriers = amplitude.mean(axis=1)

    features = {}
    features['mean'] = np.mean(mean_over_subcarriers)
    features['std'] = np.std(mean_over_subcarriers)
    features['max'] = np.max(mean_over_subcarriers)
    features['min'] = np.min(mean_over_subcarriers)
    features['range'] = features['max'] - features['min']
    features['energy'] = np.sum(mean_over_subcarriers ** 2)
    features['peak_to_average'] = features['max'] / (np.mean(np.abs(mean_over_subcarriers)) + 1e-8)

    spectrum = np.abs(rfft(mean_over_subcarriers))
    freqs = rfftfreq(len(mean_over_subcarriers), d=1 / sampling_rate)
    dominant_index = np.argmax(spectrum[1:]) + 1
    features['dominant_frequency'] = freqs[dominant_index]
    features['spectral_energy'] = np.sum(spectrum ** 2)

    return features


normal_features = extract_features(normal_amp_processed)
fall_features = extract_features(fall_amp_processed)

pd.DataFrame([normal_features, fall_features], index=['normal', 'fall_like'])

## 8. Create a synthetic dataset

This section creates multiple synthetic samples for two classes:

- `0`: normal activity
- `1`: fall-like event

These are synthetic labels for prototype testing only.

In [ ]:
def generate_synthetic_dataset(num_samples_per_class=150):
    """Generate a synthetic feature dataset for normal and fall-like classes."""
    rows = []
    labels = []

    for _ in range(num_samples_per_class):
        amp, _ = generate_normal_activity_csi(noise_level=np.random.uniform(0.03, 0.08))
        amp_processed = smooth_signal(normalize_csi(remove_mean(amp)))
        rows.append(extract_features(amp_processed))
        labels.append(0)

    for _ in range(num_samples_per_class):
        amp, _ = generate_fall_event_csi(noise_level=np.random.uniform(0.03, 0.08))
        amp_processed = smooth_signal(normalize_csi(remove_mean(amp)))
        rows.append(extract_features(amp_processed))
        labels.append(1)

    X = pd.DataFrame(rows)
    y = np.array(labels)
    return X, y


X, y = generate_synthetic_dataset(num_samples_per_class=150)

print('Feature matrix shape:', X.shape)
print('Labels shape:', y.shape)
X.head()

## 9. Train a baseline classifier

A simple Random Forest classifier is used as a baseline model. The purpose is to verify that the synthetic-data pipeline runs end to end, not to claim real-world fall-detection performance.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.25,
    random_state=RANDOM_SEED,
    stratify=y
)

model = Pipeline([
    ('scaler', StandardScaler()),
    ('classifier', RandomForestClassifier(n_estimators=100, random_state=RANDOM_SEED))
])

model.fit(X_train, y_train)
y_pred = model.predict(X_test)

accuracy = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred)
recall = recall_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)

print('Synthetic-data prototype metrics')
print('Accuracy:', round(accuracy, 3))
print('Precision:', round(precision, 3))
print('Recall:', round(recall, 3))
print('F1-score:', round(f1, 3))
print('\nClassification report:')
print(classification_report(y_test, y_pred, target_names=['normal', 'fall_like']))

## 10. Confusion matrix

The confusion matrix below summarizes the baseline classifier output on the synthetic test split.

In [ ]:
cm = confusion_matrix(y_test, y_pred)

plt.figure(figsize=(5, 4))
plt.imshow(cm)
plt.title('Baseline Confusion Matrix\nSynthetic CSI-like Data Only')
plt.xlabel('Predicted label')
plt.ylabel('True label')
plt.xticks([0, 1], ['normal', 'fall_like'])
plt.yticks([0, 1], ['normal', 'fall_like'])

for i in range(cm.shape[0]):
    for j in range(cm.shape[1]):
        plt.text(j, i, str(cm[i, j]), ha='center', va='center')

plt.colorbar()
plt.tight_layout()
plt.savefig('../figures/baseline_confusion_matrix.png', dpi=200)
plt.show()

## 11. Save baseline results

In [ ]:
results_text = f'''# Baseline Results

## Important Note

These results are based on synthetic CSI-like data and are intended only to verify the prototype pipeline. They should not be interpreted as real-world fall-detection performance, clinical validation, hardware validation, or medical-grade accuracy.

## Metrics

| Metric | Value |
|---|---:|
| Accuracy | {accuracy:.3f} |
| Precision | {precision:.3f} |
| Recall | {recall:.3f} |
| F1-score | {f1:.3f} |

## Interpretation

The baseline classifier verifies that the synthetic-data prototype pipeline can generate data, extract features, train a simple model, and produce evaluation outputs. These numbers are not evidence of real-world fall-detection performance.
'''

with open('../results/baseline_results.md', 'w', encoding='utf-8') as f:
    f.write(results_text)

print('Saved results to ../results/baseline_results.md')

## 12. Clinical-Safety-Aware Evaluation

Accuracy alone is not enough for healthcare-oriented fall detection. This section maps basic ML errors to safety-oriented prototype metrics:

- **Missed fall:** a true fall-like event predicted as normal
- **False alarm:** a normal activity sample predicted as fall-like
- **Missed fall rate:** missed falls divided by total true fall-like events
- **False alarm rate:** false alarms divided by total true normal events
- **Alarm fatigue indicator:** simple prototype indicator based on false alarm rate
- **Detection delay:** simulated delay for correctly detected fall-like events

> These metrics are computed from synthetic prototype outputs only and are not clinical validation.

In [ ]:
def compute_clinical_safety_metrics(y_true, y_pred):
    """Compute simple clinical-safety-aware metrics from binary labels."""
    y_true = np.asarray(y_true)
    y_pred = np.asarray(y_pred)

    total_falls = np.sum(y_true == 1)
    total_normal = np.sum(y_true == 0)
    missed_falls = np.sum((y_true == 1) & (y_pred == 0))
    false_alarms = np.sum((y_true == 0) & (y_pred == 1))
    detected_falls = np.sum((y_true == 1) & (y_pred == 1))

    missed_fall_rate = missed_falls / total_falls if total_falls > 0 else 0.0
    false_alarm_rate = false_alarms / total_normal if total_normal > 0 else 0.0

    if false_alarm_rate >= 0.30:
        alarm_fatigue_indicator = 'High prototype false-alarm burden'
    elif false_alarm_rate >= 0.10:
        alarm_fatigue_indicator = 'Moderate prototype false-alarm burden'
    else:
        alarm_fatigue_indicator = 'Low prototype false-alarm burden'

    return {
        'total_true_fall_like_events': int(total_falls),
        'total_true_normal_events': int(total_normal),
        'detected_falls': int(detected_falls),
        'missed_falls': int(missed_falls),
        'false_alarms': int(false_alarms),
        'missed_fall_rate': missed_fall_rate,
        'false_alarm_rate': false_alarm_rate,
        'alarm_fatigue_indicator': alarm_fatigue_indicator
    }


clinical_metrics = compute_clinical_safety_metrics(y_test, y_pred)
clinical_metrics_df = pd.DataFrame([clinical_metrics])
clinical_metrics_df

In [ ]:
correctly_detected_falls = clinical_metrics['detected_falls']

if correctly_detected_falls > 0:
    simulated_detection_delays_seconds = np.random.uniform(0.5, 4.0, size=correctly_detected_falls)
else:
    simulated_detection_delays_seconds = np.array([])

delay_summary = {
    'num_detected_falls_with_delay_estimate': int(correctly_detected_falls),
    'mean_detection_delay_seconds': float(np.mean(simulated_detection_delays_seconds)) if correctly_detected_falls > 0 else None,
    'max_detection_delay_seconds': float(np.max(simulated_detection_delays_seconds)) if correctly_detected_falls > 0 else None,
    'min_detection_delay_seconds': float(np.min(simulated_detection_delays_seconds)) if correctly_detected_falls > 0 else None
}

pd.DataFrame([delay_summary])

## 13. Save clinical-safety summary

In [ ]:
mean_delay = delay_summary['mean_detection_delay_seconds']
max_delay = delay_summary['max_detection_delay_seconds']
min_delay = delay_summary['min_detection_delay_seconds']

mean_delay_text = f'{mean_delay:.2f}' if mean_delay is not None else 'N/A'
max_delay_text = f'{max_delay:.2f}' if max_delay is not None else 'N/A'
min_delay_text = f'{min_delay:.2f}' if min_delay is not None else 'N/A'

clinical_summary_text = f'''# Clinical-Safety-Aware Summary

## Important Note

These metrics are computed from synthetic CSI-like prototype outputs only. They are intended to demonstrate the evaluation framework and should not be interpreted as clinical validation, medical-grade fall-detection performance, or real-world deployment evidence.

## Metrics

| Metric | Value |
|---|---:|
| Total true fall-like events | {clinical_metrics['total_true_fall_like_events']} |
| Total true normal events | {clinical_metrics['total_true_normal_events']} |
| Detected falls | {clinical_metrics['detected_falls']} |
| Missed falls | {clinical_metrics['missed_falls']} |
| False alarms | {clinical_metrics['false_alarms']} |
| Missed fall rate | {clinical_metrics['missed_fall_rate']:.3f} |
| False alarm rate | {clinical_metrics['false_alarm_rate']:.3f} |

## Alarm Fatigue Indicator

{clinical_metrics['alarm_fatigue_indicator']}

## Simulated Detection Delay

| Delay Metric | Value |
|---|---:|
| Mean detection delay seconds | {mean_delay_text} |
| Minimum detection delay seconds | {min_delay_text} |
| Maximum detection delay seconds | {max_delay_text} |

## Interpretation

The clinical-safety-aware metrics show how baseline ML outputs can be translated into safety-relevant indicators such as missed falls, false alarms, alarm fatigue, and detection delay.
'''

with open('../results/clinical_safety_summary.md', 'w', encoding='utf-8') as f:
    f.write(clinical_summary_text)

print('Saved clinical-safety summary to ../results/clinical_safety_summary.md')

## 14. Adversarial Robustness Stress Test

This section adds a simple synthetic robustness test. The goal is to compare clean synthetic CSI-like samples with perturbed synthetic samples.

The perturbations here are **not real physical-layer attacks**. They are synthetic stress tests that help demonstrate how future WiFi CSI security experiments could connect model degradation to safety outcomes such as missed falls and false alarms.

In [ ]:
def add_random_noise(signal, noise_std=0.25, seed=None):
    """Add Gaussian noise to a CSI-like signal."""
    rng = np.random.default_rng(seed)
    return signal + rng.normal(0, noise_std, size=signal.shape)


def add_burst_perturbation(signal, strength=1.0, center_fraction=0.55, width_fraction=0.04):
    """Add a short time-localized burst perturbation."""
    perturbed = signal.copy()
    num_packets = signal.shape[0]
    center = int(num_packets * center_fraction)
    width = max(1, int(num_packets * width_fraction))
    window = np.arange(num_packets)
    burst = strength * np.exp(-0.5 * ((window - center) / width) ** 2)
    perturbed += burst[:, None]
    return perturbed


def add_subcarrier_perturbation(signal, subcarrier_indices=None, strength=0.6):
    """Perturb selected subcarriers to simulate subcarrier-sensitive robustness testing."""
    perturbed = signal.copy()
    if subcarrier_indices is None:
        subcarrier_indices = [5, 10, 15, 20]
    for sc in subcarrier_indices:
        if sc < perturbed.shape[1]:
            perturbed[:, sc] += strength * np.sin(np.linspace(0, 8 * np.pi, perturbed.shape[0]))
    return perturbed


print('Synthetic perturbation functions defined.')

## 15. Generate clean and perturbed test sets

We create a new synthetic evaluation set, then apply perturbations before feature extraction. This allows us to compare clean and perturbed model behavior.

In [ ]:
def generate_clean_and_perturbed_feature_sets(num_samples_per_class=75, perturbation_mode='combined'):
    """Generate clean and perturbed synthetic feature sets for robustness testing."""
    clean_rows = []
    perturbed_rows = []
    labels = []

    for class_label in [0, 1]:
        for sample_idx in range(num_samples_per_class):
            if class_label == 0:
                amp, _ = generate_normal_activity_csi(noise_level=np.random.uniform(0.03, 0.08))
            else:
                amp, _ = generate_fall_event_csi(noise_level=np.random.uniform(0.03, 0.08))

            clean_amp = smooth_signal(normalize_csi(remove_mean(amp)))

            if perturbation_mode == 'noise':
                perturbed_amp = add_random_noise(clean_amp, noise_std=0.35, seed=RANDOM_SEED + sample_idx)
            elif perturbation_mode == 'burst':
                perturbed_amp = add_burst_perturbation(clean_amp, strength=1.0)
            elif perturbation_mode == 'subcarrier':
                perturbed_amp = add_subcarrier_perturbation(clean_amp, strength=0.7)
            else:
                perturbed_amp = add_random_noise(clean_amp, noise_std=0.25, seed=RANDOM_SEED + sample_idx)
                perturbed_amp = add_burst_perturbation(perturbed_amp, strength=0.8)
                perturbed_amp = add_subcarrier_perturbation(perturbed_amp, strength=0.5)

            clean_rows.append(extract_features(clean_amp))
            perturbed_rows.append(extract_features(perturbed_amp))
            labels.append(class_label)

    clean_X = pd.DataFrame(clean_rows)
    perturbed_X = pd.DataFrame(perturbed_rows)
    labels = np.array(labels)
    return clean_X, perturbed_X, labels


X_clean_eval, X_perturbed_eval, y_eval = generate_clean_and_perturbed_feature_sets(num_samples_per_class=75)

clean_pred = model.predict(X_clean_eval)
perturbed_pred = model.predict(X_perturbed_eval)

print('Clean eval shape:', X_clean_eval.shape)
print('Perturbed eval shape:', X_perturbed_eval.shape)

## 16. Compare clean vs perturbed ML metrics

In [ ]:
def summarize_ml_metrics(y_true, predictions, label):
    """Summarize common ML metrics."""
    return {
        'condition': label,
        'accuracy': accuracy_score(y_true, predictions),
        'precision': precision_score(y_true, predictions, zero_division=0),
        'recall': recall_score(y_true, predictions, zero_division=0),
        'f1_score': f1_score(y_true, predictions, zero_division=0)
    }


clean_ml_metrics = summarize_ml_metrics(y_eval, clean_pred, 'clean_synthetic')
perturbed_ml_metrics = summarize_ml_metrics(y_eval, perturbed_pred, 'perturbed_synthetic')

robustness_ml_df = pd.DataFrame([clean_ml_metrics, perturbed_ml_metrics])
robustness_ml_df

## 17. Compare clean vs perturbed safety metrics

In [ ]:
clean_safety_metrics = compute_clinical_safety_metrics(y_eval, clean_pred)
perturbed_safety_metrics = compute_clinical_safety_metrics(y_eval, perturbed_pred)

clean_safety_metrics['condition'] = 'clean_synthetic'
perturbed_safety_metrics['condition'] = 'perturbed_synthetic'

robustness_safety_df = pd.DataFrame([clean_safety_metrics, perturbed_safety_metrics])
robustness_safety_df

## 18. Visualize clean vs perturbed comparison

In [ ]:
plt.figure(figsize=(8, 5))
x_positions = np.arange(len(robustness_ml_df))
plt.bar(x_positions, robustness_ml_df['f1_score'])
plt.xticks(x_positions, robustness_ml_df['condition'], rotation=15)
plt.ylabel('F1-score')
plt.title('Clean vs Perturbed Synthetic CSI Robustness Test')
plt.ylim(0, 1.05)
plt.tight_layout()
plt.savefig('../figures/clean_vs_perturbed_f1.png', dpi=200)
plt.show()

## 19. Save adversarial robustness summary

In [ ]:
clean_row = clean_ml_metrics
perturbed_row = perturbed_ml_metrics

robustness_summary_text = f'''# Adversarial Robustness Summary

## Important Note

These results are based on synthetic perturbations applied to synthetic CSI-like data. They are intended to demonstrate a robustness-evaluation framework and should not be interpreted as evidence of a real-world physical-layer attack, clinical validation, or hardware-validated WiFi sensing performance.

## Perturbation Setup

The perturbed condition uses a combination of synthetic random noise, a burst-like disturbance, and selected subcarrier perturbations. This is a prototype stress test only.

## Clean vs Perturbed ML Metrics

| Condition | Accuracy | Precision | Recall | F1-score |
|---|---:|---:|---:|---:|
| Clean synthetic | {clean_row['accuracy']:.3f} | {clean_row['precision']:.3f} | {clean_row['recall']:.3f} | {clean_row['f1_score']:.3f} |
| Perturbed synthetic | {perturbed_row['accuracy']:.3f} | {perturbed_row['precision']:.3f} | {perturbed_row['recall']:.3f} | {perturbed_row['f1_score']:.3f} |

## Clean vs Perturbed Safety Metrics

| Condition | Missed Falls | False Alarms | Missed Fall Rate | False Alarm Rate |
|---|---:|---:|---:|---:|
| Clean synthetic | {clean_safety_metrics['missed_falls']} | {clean_safety_metrics['false_alarms']} | {clean_safety_metrics['missed_fall_rate']:.3f} | {clean_safety_metrics['false_alarm_rate']:.3f} |
| Perturbed synthetic | {perturbed_safety_metrics['missed_falls']} | {perturbed_safety_metrics['false_alarms']} | {perturbed_safety_metrics['missed_fall_rate']:.3f} | {perturbed_safety_metrics['false_alarm_rate']:.3f} |

## Interpretation

This section demonstrates how synthetic CSI perturbations can be evaluated not only with ML metrics, but also with safety-oriented metrics such as missed falls and false alarms.

The goal is to prepare a framework for future WiFi CSI security research, not to claim a real physical-layer attack implementation.

## Limitations

- Perturbations are synthetic and simplified.
- No real WiFi hardware attack is implemented.
- No real CSI dataset is used.
- No clinical validation is claimed.

## Next Steps

- Add separate experiments for noise, burst, and subcarrier perturbations.
- Evaluate robustness under increasing perturbation strengths.
- Connect attack impact to clinical-safety outcomes.
- Later evaluate with real CSI datasets or hardware experiments.
'''

with open('../results/adversarial_robustness_summary.md', 'w', encoding='utf-8') as f:
    f.write(robustness_summary_text)

print('Saved adversarial robustness summary to ../results/adversarial_robustness_summary.md')

## 20. Final limitations and next steps

This notebook is a prototype demonstration only.

### Current limitations

- The data are synthetic CSI-like signals, not real WiFi CSI measurements.
- No clinical data are used.
- No hardware deployment is included.
- The fall-like event model is simplified.
- Clinical-safety metrics are computed from synthetic labels and predictions only.
- Adversarial robustness tests use synthetic perturbations only.
- No real physical-layer attack is implemented.

### Future work

- Add real CSI datasets when available.
- Improve the synthetic signal model.
- Add stronger perturbation and adversarial testing.
- Expand clinical-safety-aware metrics.
- Connect robustness results to missed falls, false alarms, and detection delay.
- Connect this prototype to secure WiFi sensing research.